# Chapter 7: Functions as First-Class Objects
**Fluent Python, 2nd Edition** — Luciano Ramalho

---

## TL;DR

Python functions are **first-class objects**: you can assign them to variables, pass them as arguments, return them from other functions, and store them in data structures. This chapter covers higher-order functions, modern replacements for `map/filter/reduce`, lambda expressions, the nine callable types, user-defined callables via `__call__`, flexible parameter handling (keyword-only and positional-only), and the `operator` and `functools.partial` power tools.

**Key insight:** Once you treat functions as objects, many patterns (strategy, command, callback) become trivially easy — you pass functions around instead of building elaborate class hierarchies.

---

Related wiki articles:
[[first-class-functions]] | [[higher-order-functions]] | [[replacing-map-filter-reduce]] | [[anonymous-functions-lambda]] | [[callable-types]] | [[user-defined-callable-types]] | [[parameter-handling]] | [[operator-module-and-functools-partial]]

---
## 1. First-Class Functions

A **first-class object** can be:
- Created at runtime
- Assigned to a variable or data structure element
- Passed as an argument to a function
- Returned as the result of a function

In Python, **all functions** are first-class objects — integers, strings, dicts, and functions share this status equally.

See also: [[first-class-functions]]

In [ ]:
# Functions are objects — create, inspect, reassign
def factorial(n):
    """returns n!"""
    return 1 if n < 2 else n * factorial(n - 1)

# Inspect the function object
print(f"factorial(5)  = {factorial(5)}")
print(f"__doc__       = {factorial.__doc__}")
print(f"type          = {type(factorial)}")

# Assign to a variable and call through it
fact = factorial
print(f"fact(5)       = {fact(5)}")
print(f"fact is factorial: {fact is factorial}")

In [ ]:
# Pass a function as an argument
print(list(map(factorial, range(11))))

# Store functions in a data structure
func_registry = {"factorial": factorial, "hex": hex, "oct": oct}
print(func_registry["factorial"](6))  # 720

---
## 2. Higher-Order Functions

A **higher-order function** takes a function as an argument or returns a function as its result.

Common built-in examples: `sorted(key=...)`, `map()`, `filter()`, `min(key=...)`, `max(key=...)`.

See also: [[higher-order-functions]]

In [ ]:
# sorted() with a key function — a higher-order function
fruits = ["strawberry", "fig", "apple", "cherry", "raspberry", "banana"]

# Sort by length
print("By length:", sorted(fruits, key=len))

# Sort by reversed spelling (rhyme dictionary trick)
def reverse(word):
    return word[::-1]

print("By rhyme: ", sorted(fruits, key=reverse))

In [ ]:
# A function that RETURNS a function — also higher-order
def make_multiplier(factor):
    def multiplier(x):
        return x * factor
    return multiplier

double = make_multiplier(2)
triple = make_multiplier(3)
print(f"double(5) = {double(5)}")
print(f"triple(5) = {triple(5)}")

---
## 3. Modern Replacements for map, filter, and reduce

List comprehensions and generator expressions are the **Pythonic** replacement for `map()` and `filter()`. They are more readable and often faster.

`sum()` replaces the most common use of `functools.reduce()`.

See also: [[replacing-map-filter-reduce]]

In [ ]:
# map + filter vs. list comprehension
def factorial(n):
    return 1 if n < 2 else n * factorial(n - 1)

# Old style: map
print("map:     ", list(map(factorial, range(6))))
# Pythonic: listcomp
print("listcomp:", [factorial(n) for n in range(6)])

# Old style: map + filter + lambda
print("map+filter:   ", list(map(factorial, filter(lambda n: n % 2, range(6)))))
# Pythonic: listcomp with condition
print("listcomp+cond:", [factorial(n) for n in range(6) if n % 2])

In [ ]:
# reduce vs. sum
from functools import reduce
from operator import add

print("reduce:", reduce(add, range(100)))   # 4950
print("sum:   ", sum(range(100)))            # 4950  — clearer!

# Other reducing built-ins
print("all([1, 2, 3]):", all([1, 2, 3]))    # True
print("all([1, 0, 3]):", all([1, 0, 3]))    # False
print("any([0, 0, 1]):", any([0, 0, 1]))    # True
print("any([0, 0, 0]):", any([0, 0, 0]))    # False

---
## 4. Anonymous Functions (lambda)

The `lambda` keyword creates an anonymous function **limited to a single expression**.

**Best use:** throwaway callbacks passed to higher-order functions.

**Rule of thumb (Lundh's Recipe):** If a lambda is hard to read, (1) write a comment, (2) find a good name, (3) convert to `def`, (4) remove the comment.

See also: [[anonymous-functions-lambda]]

In [ ]:
# lambda as a key function
fruits = ["strawberry", "fig", "apple", "cherry", "raspberry", "banana"]
print(sorted(fruits, key=lambda word: word[::-1]))

# lambda is just syntactic sugar for def
add = lambda a, b: a + b
print(type(add))        # <class 'function'>
print(add(3, 4))        # 7
print(add.__name__)     # <lambda>  — anonymous, harder to debug

---
## 5. The Nine Flavors of Callable Objects

Python 3.9+ recognizes **nine callable types**:

| # | Type | Example |
|---|------|---------|
| 1 | User-defined functions | `def` or `lambda` |
| 2 | Built-in functions | `len`, `abs` |
| 3 | Built-in methods | `dict.get` |
| 4 | Methods | functions defined in a class body |
| 5 | Classes | calling `MyClass()` triggers `__new__` + `__init__` |
| 6 | Class instances | instances with `__call__` |
| 7 | Generator functions | functions with `yield` |
| 8 | Native coroutine functions | `async def` (Python 3.5+) |
| 9 | Async generator functions | `async def` + `yield` (Python 3.6+) |

Use `callable()` to check if an object is callable.

See also: [[callable-types]]

In [ ]:
# Demonstrating callable()
print(callable(abs))        # True  — built-in function
print(callable(str))        # True  — class (calling it creates a str)
print(callable("hello"))    # False — string is not callable
print(callable(len))        # True  — built-in function

# Check all at once
objects = [abs, str, 13, "Ni!", len, lambda x: x, type]
print([callable(obj) for obj in objects])

---
## 6. User-Defined Callable Types (`__call__`)

Implement `__call__` on a class to make its instances callable like functions. This lets you create **function-like objects with persistent state**.

See also: [[user-defined-callable-types]]

In [ ]:
import random

class BingoCage:
    """A callable that pops a random item each time it's called."""

    def __init__(self, items):
        self._items = list(items)
        random.shuffle(self._items)

    def pick(self):
        try:
            return self._items.pop()
        except IndexError:
            raise LookupError("pick from empty BingoCage")

    def __call__(self):
        """Shortcut: bingo() is the same as bingo.pick()"""
        return self.pick()

# Usage
random.seed(42)  # reproducible
bingo = BingoCage(range(5))
print("bingo.pick():", bingo.pick())
print("bingo():     ", bingo())       # __call__ delegates to pick()
print("callable?   ", callable(bingo)) # True!

In [ ]:
# Another example: a call counter decorator-like object
class CallCounter:
    """Wraps a function and counts how many times it's called."""
    def __init__(self, func):
        self.func = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        return self.func(*args, **kwargs)

@CallCounter    # This works because CallCounter instances are callable
def greet(name):
    return f"Hello, {name}!"

print(greet("Alice"))
print(greet("Bob"))
print(f"greet was called {greet.count} times")

---
## 7. Positional-Only and Keyword-Only Parameters

Python gives you fine-grained control over how callers pass arguments:

- **Keyword-only** parameters: appear **after** `*` in the signature.
- **Positional-only** parameters: appear **before** `/` in the signature (Python 3.8+).

```
def f(pos_only, /, normal, *, kw_only):
    ...
```

See also: [[parameter-handling]]

In [ ]:
# The tag() example — demonstrates *content, keyword-only, and **attrs
def tag(name, /, *content, class_=None, **attrs):
    """Generate one or more HTML tags."""
    if class_ is not None:
        attrs["class"] = class_
    attr_pairs = (f' {attr}="{value}"' for attr, value
                  in sorted(attrs.items()))
    attr_str = "".join(attr_pairs)
    if content:
        elements = (f"<{name}{attr_str}>{c}</{name}>" for c in content)
        return "\n".join(elements)
    else:
        return f"<{name}{attr_str} />"

# Various calling patterns
print(tag("br"))                                    # positional only
print(tag("p", "hello"))                            # *content captures "hello"
print(tag("p", "hello", "world"))                   # multiple *content
print(tag("p", "hello", id=33))                     # **attrs captures id
print(tag("p", "hello", class_="sidebar"))          # keyword-only class_

In [ ]:
# Keyword-only parameters: anything after * or *args
def f(a, *, b):
    return a, b

print(f(1, b=2))      # OK
try:
    f(1, 2)            # Fails — b is keyword-only
except TypeError as e:
    print(f"Error: {e}")

# Positional-only parameters: anything before /  (Python 3.8+)
def divmod_custom(a, b, /):
    return (a // b, a % b)

print(divmod_custom(10, 3))    # OK
try:
    divmod_custom(a=10, b=3)   # Fails — a, b are positional-only
except TypeError as e:
    print(f"Error: {e}")

---
## 8. The operator Module and functools.partial

### operator module
Provides function equivalents of operators, eliminating trivial lambdas:
- `operator.mul`, `operator.add` — arithmetic
- `operator.itemgetter` — pick items by index/key
- `operator.attrgetter` — pick attributes by name
- `operator.methodcaller` — call a method by name

### functools.partial
Freezes some arguments of a callable, producing a **new callable** with a simpler signature.

See also: [[operator-module-and-functools-partial]]

In [ ]:
# operator.mul replaces lambda a, b: a * b
from functools import reduce
from operator import mul

def factorial_reduce(n):
    return reduce(mul, range(1, n + 1))

print(f"10! = {factorial_reduce(10)}")

In [ ]:
# itemgetter — sort tuples by field, replace lambda
from operator import itemgetter

metro_data = [
    ("Tokyo", "JP", 36.933, (35.689722, 139.691667)),
    ("Delhi NCR", "IN", 21.935, (28.613889, 77.208889)),
    ("Mexico City", "MX", 20.142, (19.433333, -99.133333)),
    ("New York-Newark", "US", 20.104, (40.808611, -74.020386)),
    ("Sao Paulo", "BR", 19.649, (-23.547778, -46.635833)),
]

# Sort by country code (index 1)
for city in sorted(metro_data, key=itemgetter(1)):
    print(city)

print()

# Extract multiple fields
cc_name = itemgetter(1, 0)
for city in metro_data:
    print(cc_name(city))

In [ ]:
# attrgetter — extract attributes, even nested ones
from collections import namedtuple
from operator import attrgetter

LatLon = namedtuple("LatLon", "lat lon")
Metropolis = namedtuple("Metropolis", "name cc pop coord")

metro_areas = [Metropolis(name, cc, pop, LatLon(lat, lon))
               for name, cc, pop, (lat, lon) in metro_data]

# Sort by nested attribute coord.lat
name_lat = attrgetter("name", "coord.lat")
for city in sorted(metro_areas, key=attrgetter("coord.lat")):
    print(name_lat(city))

In [ ]:
# methodcaller — call a named method
from operator import methodcaller

s = "The time has come"
upcase = methodcaller("upper")
print(upcase(s))

hyphenate = methodcaller("replace", " ", "-")
print(hyphenate(s))

In [ ]:
# functools.partial — freeze arguments
from functools import partial
from operator import mul

triple = partial(mul, 3)
print(f"triple(7) = {triple(7)}")
print(f"map(triple, 1..9) = {list(map(triple, range(1, 10)))}")

# Practical: Unicode normalization shortcut
import unicodedata
nfc = partial(unicodedata.normalize, "NFC")
s1 = "caf\u00e9"          # e-with-acute
s2 = "cafe\u0301"         # e + combining acute
print(f"s1 == s2:      {s1 == s2}")       # False
print(f"nfc(s1)==nfc(s2): {nfc(s1) == nfc(s2)}")  # True

# Inspect partial attributes
print(f"triple.func:     {triple.func}")
print(f"triple.args:     {triple.args}")
print(f"triple.keywords: {triple.keywords}")

---
## Exercises

Test your understanding of first-class functions.

### Exercise 1: Higher-Order Function

Write a function `apply_twice(func, value)` that applies `func` to `value` two times (i.e., `func(func(value))`).

Test it with:
- `apply_twice(lambda x: x + 3, 10)` should return 16
- `apply_twice(str.upper, "hello")` should return `"HELLO"`

In [ ]:
# Exercise 1: Your solution here
def apply_twice(func, value):
    return func(func(value))

# Tests
assert apply_twice(lambda x: x + 3, 10) == 16
assert apply_twice(str.upper, "hello") == "HELLO"
print("Exercise 1 passed!")

### Exercise 2: Replace lambda with operator

Rewrite this code to use `operator.itemgetter` instead of a lambda:

```python
students = [("Alice", 92), ("Bob", 85), ("Charlie", 95)]
sorted(students, key=lambda s: s[1])
```

In [ ]:
# Exercise 2: Your solution here
from operator import itemgetter

students = [("Alice", 92), ("Bob", 85), ("Charlie", 95)]
by_grade = sorted(students, key=itemgetter(1))
print(by_grade)
assert by_grade == [("Bob", 85), ("Alice", 92), ("Charlie", 95)]
print("Exercise 2 passed!")

### Exercise 3: Build a Callable Class

Create a class `Averager` that keeps a running average. Each call adds a new value and returns the current average.

```python
avg = Averager()
avg(10)  # -> 10.0
avg(20)  # -> 15.0
avg(30)  # -> 20.0
```

In [ ]:
# Exercise 3: Your solution here
class Averager:
    def __init__(self):
        self._values = []

    def __call__(self, value):
        self._values.append(value)
        return sum(self._values) / len(self._values)

avg = Averager()
assert avg(10) == 10.0
assert avg(20) == 15.0
assert avg(30) == 20.0
print("Exercise 3 passed!")

### Exercise 4: functools.partial in Practice

Use `functools.partial` to create a `base2` function from `int()` that converts binary strings to integers (i.e., `int(x, base=2)`).

Test: `base2("1010")` should return 10.

In [ ]:
# Exercise 4: Your solution here
from functools import partial

base2 = partial(int, base=2)
assert base2("1010") == 10
assert base2("11111111") == 255
assert base2("0") == 0
print("base2('1010') =", base2("1010"))
print("Exercise 4 passed!")

### Exercise 5: Keyword-Only Parameters

Write a function `clip(text, *, max_len=80)` where `max_len` is keyword-only. It should truncate `text` to `max_len` characters, appending `"..."` if truncated.

`clip("Hello World", max_len=5)` should return `"He..."`

In [ ]:
# Exercise 5: Your solution here
def clip(text, *, max_len=80):
    if len(text) <= max_len:
        return text
    return text[:max_len - 3] + "..."

assert clip("Hello World", max_len=5) == "He..."
assert clip("Hi", max_len=80) == "Hi"
assert clip("ABCDEFGH", max_len=6) == "ABC..."
print("Exercise 5 passed!")

---
## Key Takeaways

1. **Functions are objects.** You can assign, pass, return, and introspect them just like any other Python object.

2. **Prefer listcomps over map/filter.** `[f(x) for x in items if cond(x)]` beats `list(map(f, filter(cond, items)))` every time for readability.

3. **Keep lambdas simple.** If it does not fit on one line or needs a comment, use `def`.

4. **`callable()` is your friend.** Nine types of objects are callable; use `callable()` to check.

5. **`__call__` bridges functions and objects.** Implement it to give instances function-like behavior with persistent state.

6. **Use `/` and `*` in signatures** to make your APIs precise: positional-only for internal parameters, keyword-only for flags and options.

7. **`operator` and `functools.partial`** eliminate most trivial lambdas and make functional-style code cleaner.

---

## Navigation

| Previous | Up | Next |
|----------|-----|------|
| [[Chapter 06]] | [[Fluent Python]] | [[Chapter 08]] |

---

*Generated for the distill-tech-books knowledge base.*